# Integrantes de la Tarea 1:

## Fernando Cabrera Legue - Sección 1.

Desarrollo Parte 2 de la tarea.

## Nhayara Ramos Caceres - Sección 2.

Desarrollo Parte 1 y 2 de la tarea.

# Parte 1 - Redes Bayesianas

## 1.1 Carga y exploración del dataset

Para esta parte se utilizará el dataset Adult de UCI Machine Learning Repository.


In [46]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

adult = fetch_ucirepo(id=2)

X = adult.data.features
y = adult.data.targets

df = pd.concat([X, y], axis=1)

In [47]:
df.shape

(48842, 15)

In [48]:
df.columns

Index(['age', 'workclass', 'fnlwgt', 'education', 'education-num',
       'marital-status', 'occupation', 'relationship', 'race', 'sex',
       'capital-gain', 'capital-loss', 'hours-per-week', 'native-country',
       'income'],
      dtype='str')

Las variables incluyen información personal, laboral y educacional. La variable objetivo es `income`, que indica si una persona gana más o menos de 50K.

In [49]:
df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


### Revisión inicial

Antes de construir la red bayesiana, se revisaron las dimensiones, las columnas y la presencia de valores faltantes para preparar correctamente los datos.

Con `df.info()` se revisaron los tipos de datos de cada variable y la cantidad de registros no nulos presentes en cada columna.

In [50]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             48842 non-null  int64
 1   workclass       47879 non-null  str  
 2   fnlwgt          48842 non-null  int64
 3   education       48842 non-null  str  
 4   education-num   48842 non-null  int64
 5   marital-status  48842 non-null  str  
 6   occupation      47876 non-null  str  
 7   relationship    48842 non-null  str  
 8   race            48842 non-null  str  
 9   sex             48842 non-null  str  
 10  capital-gain    48842 non-null  int64
 11  capital-loss    48842 non-null  int64
 12  hours-per-week  48842 non-null  int64
 13  native-country  48568 non-null  str  
 14  income          48842 non-null  str  
dtypes: int64(6), str(9)
memory usage: 5.6 MB


In [51]:
df.isnull().sum()

age                 0
workclass         963
fnlwgt              0
education           0
education-num       0
marital-status      0
occupation        966
relationship        0
race                0
sex                 0
capital-gain        0
capital-loss        0
hours-per-week      0
native-country    274
income              0
dtype: int64

Se encontraron valores faltantes únicamente en `workclass`, `occupation` y `native-country`.

Como la cantidad de registros incompletos es pequeña en comparación con el total del dataset, se decidió eliminar esas filas antes de continuar con el análisis.

In [52]:
df_sin_nulos = df.dropna()

print("Filas originales:", len(df))
print("Filas sin datos faltantes:", len(df_sin_nulos))
print("Filas eliminadas:", len(df) - len(df_sin_nulos))

Filas originales: 48842
Filas sin datos faltantes: 47621
Filas eliminadas: 1221


Luego de eliminar los registros con valores faltantes, el dataset quedó con **47.621 registros completos**, eliminándose **1.221 filas**.

### Revisión de categorías

Se revisó la cantidad de valores distintos de cada variable para identificar posibles inconsistencias en las categorías.

In [53]:
df.nunique()

age                  74
workclass             9
fnlwgt            28523
education            16
education-num        16
marital-status        7
occupation           15
relationship          6
race                  5
sex                   2
capital-gain        123
capital-loss         99
hours-per-week       96
native-country       42
income                4
dtype: int64

In [54]:
df["income"].value_counts()
df["income"].unique()

<StringArray>
['<=50K', '>50K', '<=50K.', '>50K.']
Length: 4, dtype: str

En la variable objetivo `income` se encontraron cuatro etiquetas distintas:

- `<=50K`
- `>50K`
- `<=50K.`
- `>50K.`

Sin embargo, estas etiquetas representan solo dos categorías reales. La diferencia corresponde únicamente al punto final presente en algunos registros.

In [55]:
df["income"].value_counts()

income
<=50K     24720
<=50K.    12435
>50K       7841
>50K.      3846
Name: count, dtype: int64

In [56]:
df["income"] = df["income"].str.replace(".", "", regex=False)

In [57]:
df["income"].value_counts()

income
<=50K    37155
>50K     11687
Name: count, dtype: int64

In [58]:
df = df.dropna().copy()

In [59]:
print("Dimensiones después de eliminar nulos:", df.shape)
print("\nValores faltantes:")
print(df.isnull().sum())

Dimensiones después de eliminar nulos: (47621, 15)

Valores faltantes:
age               0
workclass         0
fnlwgt            0
education         0
education-num     0
marital-status    0
occupation        0
relationship      0
race              0
sex               0
capital-gain      0
capital-loss      0
hours-per-week    0
native-country    0
income            0
dtype: int64


### Limpieza de datos

Se identificaron valores faltantes en las variables `workclass`, `occupation` y
`native-country`. Se eliminaron los registros que contenían valores faltantes,
quedando 47.621 observaciones completas.

Además, la variable objetivo `income` presentaba categorías equivalentes con
diferencias de formato (`<=50K` y `<=50K.`, `>50K` y `>50K.`), por lo que estas
etiquetas fueron unificadas.


### Selección de variables

Se revisaron las variables disponibles y se eliminaron aquellas que se consideraron insignificantes para el modelo propuesto 

Se eliminó la variable `fnlwgt`, ya que corresponde a un peso estadístico asociado a cada registro y lo consideramos insignificante para el modelo propuesto aparte que anda cambiando a cada rato 

También se eliminó la variable `education-num`, debido a que corresponde a una codificación numérica del nivel educativo y representa información que ya se encuentra en la variable `education`.


In [60]:
df = df.drop(columns=["fnlwgt", "education-num"])

print("Tamaño  del dataset:", df.shape)
print("\nColumnas seleccionadas:")
print(df.columns)

Tamaño  del dataset: (47621, 13)

Columnas seleccionadas:
Index(['age', 'workclass', 'education', 'marital-status', 'occupation',
       'relationship', 'race', 'sex', 'capital-gain', 'capital-loss',
       'hours-per-week', 'native-country', 'income'],
      dtype='str')


In [61]:
print(df.dtypes)

age               int64
workclass           str
education           str
marital-status      str
occupation          str
relationship        str
race                str
sex                 str
capital-gain      int64
capital-loss      int64
hours-per-week    int64
native-country      str
income              str
dtype: object


Nos fijamos solo en los enteros

In [62]:
df[["age", "capital-gain", "capital-loss", "hours-per-week"]].describe()

,age,capital-gain,capital-loss,hours-per-week
count,47621.000000,47621.000000,47621.000000,47621.000000
mean,38.640684,1091.137649,87.853489,40.600050
std,13.558961,7487.228336,404.010612,12.260345
min,17.000000,0.000000,0.000000,1.000000
25%,28.000000,0.000000,0.000000,40.000000
50%,37.000000,0.000000,0.000000,40.000000
75%,48.000000,0.000000,0.000000,45.000000
max,90.000000,99999.000000,4356.000000,99.000000


In [63]:
print(df["age"].value_counts().sort_index())

age
17     560
18     798
19     982
20    1040
21    1034
      ... 
86       1
87       2
88       5
89       1
90      54
Name: count, Length: 74, dtype: int64


### Discretización de la variable `age`

La variable `age` presentaba 74 valores diferentes, correspondientes a edades entre 17 y 90 años. Para reducir la cantidad de estados de esta variable y facilitarla en una Red Bayesiana discreta, se decidió discretizarla.

Se utilizó una división basada en cuartiles mediante `pd.qcut()`, separando las observaciones en cuatro grupos. De esta manera, los grupos se determinan a partir de la distribución de las edades presentes en el dataset, en lugar de establecer los límites de forma arbitraria.

In [64]:
df["age"] = pd.qcut(
    df["age"],
    q=4,
    labels=["Edad_1", "Edad_2", "Edad_3", "Edad_4"]
)

In [65]:
print(df["age"].value_counts().sort_index())

age
Edad_1    12776
Edad_2    11514
Edad_3    12194
Edad_4    11137
Name: count, dtype: int64


In [66]:
print(df["age"].cat.categories)

Index(['Edad_1', 'Edad_2', 'Edad_3', 'Edad_4'], dtype='str')


In [67]:
print("Capital gain = 0:")
print((df["capital-gain"] == 0).sum())

print("\nCapital gain > 0:")
print((df["capital-gain"] > 0).sum())

Capital gain = 0:
43657

Capital gain > 0:
3964


In [68]:
df.loc[df["capital-gain"] > 0, "capital-gain"].describe()

count     3964.000000
mean     13108.240666
std      22716.635811
min        114.000000
25%       3411.000000
50%       7298.000000
75%      13550.000000
max      99999.000000
Name: capital-gain, dtype: float64

### Discretización de la variable `capital-gain`

La variable `capital-gain` presenta una alta concentración de valores iguales a cero. 

Debido a esta distribución, no se utilizó la misma discretización por cuartiles aplicada a `age`. Se decidió mantener el valor cero como una categoria independiente denominada `Sin_ganancia` y dividir los valores positivos en tres niveles: `Baja`, `Media` y `Alta`.

Para determinar los límites entre estos tres niveles se utilizaran los valores positivos de `capital-gain`, permitiendo obtener los puntos de corte a partir de la distribución de los propios datos.

In [69]:
positivos = df.loc[df["capital-gain"] > 0, "capital-gain"]

cortes_gain = positivos.quantile([1/3, 2/3])

print(cortes_gain)

0.333333    4386.0
0.666667    7688.0
Name: capital-gain, dtype: float64


In [70]:
def discretizar_gain(valor):
    if valor == 0:
        return "Sin_ganancia"
    elif valor <= 4386:
        return "Baja"
    elif valor <= 7688:
        return "Media"
    else:
        return "Alta"

df["capital-gain"] = df["capital-gain"].apply(discretizar_gain)

In [71]:
print(df["capital-gain"].value_counts())

capital-gain
Sin_ganancia    43657
Media            1362
Baja             1357
Alta             1245
Name: count, dtype: int64


In [72]:
print("Capital loss = 0:")
print((df["capital-loss"] == 0).sum())

print("\nCapital loss > 0:")
print((df["capital-loss"] > 0).sum())

Capital loss = 0:
45389

Capital loss > 0:
2232


### Discretización de la variable `capital-loss`

La variable `capital-loss` presenta una alta concentración de valores iguales a cero. Para ello  se decidio separar el valor cero en una categoría denominada `Sin_perdida` y estudiar los valores positivos por separado.

Al igual que con `capital-gain`, los valores positivos serán divididos en tres niveles: `Baja`, `Media` y `Alta`. Los puntos de corte se obtendrán a partir de los terciles de los valores positivo.

In [73]:
positivos_loss = df.loc[df["capital-loss"] > 0, "capital-loss"]

cortes_loss = positivos_loss.quantile([1/3, 2/3])

print(cortes_loss)

0.333333    1755.0
0.666667    1977.0
Name: capital-loss, dtype: float64


In [74]:
def discretizar_loss(valor):
    if valor == 0:
        return "Sin_perdida"
    elif valor <= 1755:
        return "Baja"
    elif valor <= 1977:
        return "Media"
    else:
        return "Alta"

df["capital-loss"] = df["capital-loss"].apply(discretizar_loss)

In [75]:
print(df["capital-loss"].value_counts())

capital-loss
Sin_perdida    45389
Media            971
Baja             745
Alta             516
Name: count, dtype: int64


In [76]:
print("Menos de 40 horas:")
print((df["hours-per-week"] < 40).sum())

print("\nExactamente 40 horas:")
print((df["hours-per-week"] == 40).sum())

print("\nMas de 40 horas:")
print((df["hours-per-week"] > 40).sum())

Menos de 40 horas:
11136

Exactamente 40 horas:
22324

Mas de 40 horas:
14161


### Discretización de la variable `hours-per-week`

La variable `hours-per-week` presenta una alta concentración de observaciones en 40 horas semanales. 

Debido a esta distribución, se decidió utilizar 40 horas como punto de referencia y dividir la variable en tres categorias: `Menos_40`, `40_horas` y `Mas_40`.

In [77]:
def discretizar_horas(valor):
    if valor < 40:
        return "Menos_40"
    elif valor == 40:
        return "40_horas"
    else:
        return "Mas_40"

df["hours-per-week"] = df["hours-per-week"].apply(discretizar_horas)

In [78]:
print(df["hours-per-week"].value_counts())

hours-per-week
40_horas    22324
Mas_40      14161
Menos_40    11136
Name: count, dtype: int64


Comrpobamos los estados de cada variable 

In [79]:
print("Cantidad de estados por variable:\n")

for columna in df.columns:
    print(columna, ":", df[columna].nunique())

Cantidad de estados por variable:

age : 4
workclass : 9
education : 16
marital-status : 7
occupation : 15
relationship : 6
race : 5
sex : 2
capital-gain : 4
capital-loss : 4
hours-per-week : 3
native-country : 42
income : 2


Native-country tiene muchos estados , esto puede hacer que la red bayesiana , su tabla de probabilidades sea demasiado grande

In [80]:
print(df["native-country"].value_counts())

native-country
United-States                 42958
Mexico                          936
?                               583
Philippines                     293
Germany                         202
Puerto-Rico                     180
Canada                          177
El-Salvador                     153
India                           147
Cuba                            136
England                         123
China                           120
South                           110
Italy                           105
Jamaica                         104
Dominican-Republic              100
Japan                            92
Guatemala                        87
Vietnam                          86
Poland                           85
Columbia                         85
Haiti                            71
Portugal                         65
Taiwan                           64
Iran                             57
Greece                           49
Nicaragua                        49
Peru         

In [81]:
for columna in df.columns:
    cantidad = (df[columna] == "?").sum()
    if cantidad > 0:
        print(columna, ":", cantidad)

workclass : 1836
occupation : 1843
native-country : 583


### Tratamiento de valores desconocidos

Durante la revisión de las variables categóricas se identificaron registros con el valor `?` en las variables `workclass`, `occupation` y `native-country`.

Estos valores representan información desconocida y no habían sido eliminados anteriormente mediante `dropna()`, ya que se encuentran almacenados como texto. Para mantener el criterio de trabajar con observaciones completas, se decidió eliminar los registros que contienen `?`.

In [82]:
df = df[~df.isin(["?"]).any(axis=1)].copy()

print("Dimensiones después de eliminar '?':", df.shape)

Dimensiones después de eliminar '?': (45222, 13)


In [83]:
for columna in df.columns:
    cantidad = (df[columna] == "?").sum()
    if cantidad > 0:
        print(columna, ":", cantidad)

In [84]:
print("Cantidad de países:", df["native-country"].nunique())
print("\nFrecuencia por país:")
print(df["native-country"].value_counts())

Cantidad de países: 41

Frecuencia por país:
native-country
United-States                 41292
Mexico                          903
Philippines                     283
Germany                         193
Puerto-Rico                     175
Canada                          163
India                           147
El-Salvador                     147
Cuba                            133
England                         119
China                           113
Jamaica                         103
South                           101
Italy                           100
Dominican-Republic               97
Japan                            89
Guatemala                        86
Vietnam                          83
Columbia                         82
Poland                           81
Haiti                            69
Portugal                         62
Iran                             56
Taiwan                           55
Greece                           49
Nicaragua                        48
Peru

### Agrupación de la variable `native-country`

La variable `native-country` presentaba 41 categorías diferentes. Además, se observó una alta concentración de registros correspondientes a `United-States`, mientras que los demás países presentaban una frecuencia mas pequeña que digamos .

Para reducir la cantidad de estados de esta variable y facilitar su utilización en la Red Bayesiana, se decidió agruparla en dos categorías: `United-States` para los registros pertenecientes a Estados Unidos y `Otro_pais` para el resto de los países.

In [85]:
df["native-country"] = df["native-country"].apply(
    lambda pais: "United-States" if pais == "United-States" else "Otro_pais"
)

In [86]:
print(df["native-country"].value_counts())

native-country
United-States    41292
Otro_pais         3930
Name: count, dtype: int64


In [87]:
print("Por finnn , parte final :", df.shape)

print("\nCantidad de estados por variable:")
for columna in df.columns:
    print(columna, ":", df[columna].nunique())

Por finnn , parte final : (45222, 13)

Cantidad de estados por variable:
age : 4
workclass : 7
education : 16
marital-status : 7
occupation : 14
relationship : 6
race : 5
sex : 2
capital-gain : 4
capital-loss : 4
hours-per-week : 3
native-country : 2
income : 2


### Descripción de las variables utilizadas

Antes de definir la estructura de la Red Bayesiana, se revisó el significado de las variables seleccionadas. Esto permite comprender la información que representa cada una y posteriormente establecer las relaciones entre ellas.

| Variable | Significado | Descripción |
|---|---|---|
| `age` | Edad | Edad de la persona, discretizada en cuatro grupos. |
| `workclass` | Tipo de empleo | Sector o tipo de organización en la que trabaja la persona, como empresa privada, gobierno o trabajo independiente. |
| `education` | Nivel educacional | Máximo nivel de educación alcanzado por la persona. |
| `marital-status` | Estado civil | Estado civil de la persona, como casado, soltero, divorciado o viudo. |
| `occupation` | Ocupación | Tipo de trabajo o actividad laboral realizada por la persona. |
| `relationship` | Relación en el hogar | Rol o relación que posee la persona dentro de su hogar. |
| `race` | Categoría racial | Categoría racial registrada para la persona en el dataset. |
| `sex` | Sexo | Sexo registrado de la persona. |
| `capital-gain` | Ganancia de capital | Ganancias de capital registradas, discretizadas en Sin ganancia, Baja, Media y Alta. |
| `capital-loss` | Pérdida de capital | Pérdidas de capital registradas, discretizadas en Sin pérdida, Baja, Media y Alta. |
| `hours-per-week` | Horas trabajadas por semana | Cantidad de horas que la persona trabaja semanalmente, discretizada en menos de 40, 40 horas y más de 40. |
| `native-country` | País de origen | País de origen registrado, agrupado en Estados Unidos y Otro país. |
| `income` | Ingreso anual | Variable objetivo que indica si el ingreso anual es `<=50K` o `>50K`. |

## 1.2 Propuesta de estructura de la Red Bayesiana

 teniendo a `income` como la variable objetivo del modelo , se propone la siguiente red 

Las dependencias propuestas son:

| Variable origen | Variable dependiente | Justificación |
|---|---|---|
| `education` | `workclass` | Se considera que el nivel educacional puede estar asociado con el tipo o sector de empleo de una persona. |
| `sex` | `occupation` | Se considera que la distribución de las ocupaciones puede variar según el sexo registrado en el dataset. |
| `native-country` | `race` | Se considera que la distribución de las categorías de `race` puede variar según el grupo de país de origen registrado. |
| `age` | `hours-per-week` | Se considera que la cantidad de horas trabajadas semanalmente puede variar entre los distintos grupos de edad. |
| `age` | `marital-status` | Se considera que la distribución del estado civil puede variar según el grupo de edad. |
| `age` | `relationship` | Se considera que el rol de una persona dentro del hogar puede variar según su grupo de edad. |
| `occupation` | `capital-gain` | Se considera que las ganancias de capital pueden presentar distribuciones diferentes según el tipo de ocupación. |
| `education` | `income` | El nivel educacional puede aportar información para estimar la categoría de ingreso anual. |
| `occupation` | `income` | El tipo de ocupación puede aportar información para estimar la categoría de ingreso anual. |
| `hours-per-week` | `income` | Las horas trabajadas semanalmente pueden aportar información para estimar la categoría de ingreso anual. |
| `capital-gain` | `income` | Las ganancias de capital pueden aportar información para estimar la categoría de ingreso anual. |
| `capital-loss` | `income` | Las pérdidas de capital pueden aportar información para estimar la categoría de ingreso anual. |

Estas relaciones representan dependencias probabilísticas propuestas para el modelo y no necesariamente relaciones causales.

In [88]:
from pgmpy.models import DiscreteBayesianNetwork

modelo_manual = DiscreteBayesianNetwork([
    ("education", "workclass"),
    ("sex", "occupation"),
    ("native-country", "race"),
    ("age", "hours-per-week"),
    ("age", "marital-status"),
    ("age", "relationship"),
    ("occupation", "capital-gain"),
    ("education", "income"),
    ("occupation", "income"),
    ("hours-per-week", "income"),
    ("capital-gain", "income"),
    ("capital-loss", "income")
])

In [89]:
import networkx as nx

print("Nodos:")
print(modelo_manual.nodes())

print("\nRelaciones:")
print(modelo_manual.edges())

print("\n¿Es un DAG válido?:")
print(nx.is_directed_acyclic_graph(modelo_manual))

Nodos:
['education', 'workclass', 'sex', 'occupation', 'native-country', 'race', 'age', 'hours-per-week', 'marital-status', 'relationship', 'capital-gain', 'income', 'capital-loss']

Relaciones:
[('education', 'workclass'), ('education', 'income'), ('sex', 'occupation'), ('occupation', 'capital-gain'), ('occupation', 'income'), ('native-country', 'race'), ('age', 'hours-per-week'), ('age', 'marital-status'), ('age', 'relationship'), ('hours-per-week', 'income'), ('capital-gain', 'income'), ('capital-loss', 'income')]

¿Es un DAG válido?:
True


## 1.3 Aprendizaje de parámetros

Una vez definida la estructura manual de la Red Bayesiana, se procede a estimar sus parametros utilizando las observaciones del dataset preparado.

In [90]:
from pgmpy.estimators import MaximumLikelihoodEstimator

estimador = MaximumLikelihoodEstimator(modelo_manual, df)

cpds = estimador.get_parameters()

modelo_manual.add_cpds(*cpds)

/tmp/ipykernel_223051/2119084374.py:3: FutureWarning: `pgmpy.estimators.MaximumLikelihoodEstimator` is deprecated and will be removed in v1.3.0. Please use `pgmpy.parameter_estimator.DiscreteMLE` instead.
  estimador = MaximumLikelihoodEstimator(modelo_manual, df)


In [91]:
print("¿Modelo válido?:", modelo_manual.check_model())
print("Cantidad de CPDs:", len(modelo_manual.get_cpds()))

¿Modelo válido?: True
Cantidad de CPDs: 13


In [92]:
print(modelo_manual.get_cpds("income"))

+----------------+-----+------------------------------+
| capital-gain   | ... | capital-gain(Sin_ganancia)   |
+----------------+-----+------------------------------+
| capital-loss   | ... | capital-loss(Sin_perdida)    |
+----------------+-----+------------------------------+
| education      | ... | education(Some-college)      |
+----------------+-----+------------------------------+
| hours-per-week | ... | hours-per-week(Menos_40)     |
+----------------+-----+------------------------------+
| occupation     | ... | occupation(Transport-moving) |
+----------------+-----+------------------------------+
| income(<=50K)  | ... | 0.9230769230769231           |
+----------------+-----+------------------------------+
| income(>50K)   | ... | 0.07692307692307693          |
+----------------+-----+------------------------------+


In [93]:
from pgmpy.inference import VariableElimination

inferencia_manual = VariableElimination(modelo_manual)

In [94]:
resultado = inferencia_manual.query(
    variables=["income"],
    evidence={
        "education": "Bachelors",
        "hours-per-week": "Mas_40"
    }
)

print(resultado)

+---------------+---------------+
| income        |   phi(income) |
+===============+===============+
| income(<=50K) |        0.5731 |
+---------------+---------------+
| income(>50K)  |        0.4269 |
+---------------+---------------+


¿Cuál es la probabilidad del nivel de ingreso si sabemos que la persona tiene nivel educacional Bachelors y trabaja más de 40 horas semanales?

### Inferencia 1

**Evidencia considerada:**
- `education = Bachelors`
- `hours-per-week = Mas_40`

La inferencia obtenida fue:

- $P(income \leq 50K) = 0.5731$
- $P(income > 50K) = 0.4269$

Por lo tanto, bajo esta evidencia, el modelo asigna una probabilidad de **57,31 %** a un ingreso menor o igual a 50K y una probabilidad de **42,69 %** a un ingreso superior a 50K.

In [95]:
resultado2 = inferencia_manual.query(
    variables=["income"],
    evidence={
        "age": "Edad_1"
    }
)

print(resultado2)

+---------------+---------------+
| income        |   phi(income) |
+===============+===============+
| income(<=50K) |        0.7872 |
+---------------+---------------+
| income(>50K)  |        0.2128 |
+---------------+---------------+


¿Cuál es la probabilidad de pertenecer al grupo >50K sabiendo que la persona está en cierto grupo de edad?

### Inferencia 2


**Evidencia considerada:**
- `age = Edad_1`

La inferencia obtenida fue:

- $P(income \leq 50K) = 0.7872$
- $P(income > 50K) = 0.2128$

Por lo tanto, para una persona perteneciente al grupo `Edad_1`, el modelo estima una probabilidad de **78,72 %** de pertenecer al grupo de ingreso menor o igual a 50K y una probabilidad de **21,28 %** de pertenecer al grupo de ingreso superior a 50K.

In [96]:
resultado3 = inferencia_manual.query(
    variables=["hours-per-week"],
    evidence={
        "age": "Edad_4"
    }
)

print(resultado3)

+--------------------------+-----------------------+
| hours-per-week           |   phi(hours-per-week) |
+==========================+=======================+
| hours-per-week(40_horas) |                0.4652 |
+--------------------------+-----------------------+
| hours-per-week(Mas_40)   |                0.3016 |
+--------------------------+-----------------------+
| hours-per-week(Menos_40) |                0.2331 |
+--------------------------+-----------------------+


¿Cuál es la probabilidad de trabajar más de 40 horas semanales sabiendo que la persona pertenece al grupo de mayor edad (Edad_4)?

### Inferencia 3

**Evidencia considerada:**
- `age = Edad_4`

La inferencia obtenida fue:

- $P(hours\text{-}per\text{-}week = 40\_horas) = 0.4652$
- $P(hours\text{-}per\text{-}week = Mas\_40) = 0.3016$
- $P(hours\text{-}per\text{-}week = Menos\_40) = 0.2331$

Por lo tanto, para una persona perteneciente al grupo `Edad_4`, el modelo estima una probabilidad de **30,16 % de trabajar más de 40 horas semanales**. La categoría con mayor probabilidad corresponde a trabajar exactamente 40 horas semanales, con un **46,52 %**.

## 1.4 Aprendizaje automático de una segunda estructura

Para construir una segunda Red Bayesiana se utilizará un algoritmo de aprendizaje de estructura disponible en `pgmpy`.

Se utilizará Hill Climb Search, un método basado en puntuación que modifica iterativamente la estructura de la red buscando una configuración que presente un mejor ajuste a los datos.

In [97]:
from pgmpy.causal_discovery import HillClimbSearch

hc = HillClimbSearch(
    scoring_method="bic-d",
    show_progress=False
)

hc.fit(df)

estructura_auto = hc.causal_graph_

In [98]:
print("Nodos:")
print(estructura_auto.nodes())

print("\nRelaciones aprendidas automáticamente:")
print(estructura_auto.edges())

print("\nCantidad de relaciones:")
print(len(estructura_auto.edges()))

Nodos:
['workclass', 'occupation', 'sex', 'education', 'age', 'marital-status', 'relationship', 'capital-gain', 'race', 'native-country', 'capital-loss', 'income', 'hours-per-week']

Relaciones aprendidas automáticamente:
[('workclass', 'occupation'), ('workclass', 'age'), ('occupation', 'education'), ('occupation', 'hours-per-week'), ('sex', 'occupation'), ('sex', 'relationship'), ('sex', 'marital-status'), ('education', 'native-country'), ('education', 'income'), ('age', 'marital-status'), ('age', 'workclass'), ('age', 'relationship'), ('marital-status', 'sex'), ('marital-status', 'relationship'), ('marital-status', 'age'), ('relationship', 'capital-gain'), ('relationship', 'sex'), ('relationship', 'race'), ('relationship', 'marital-status'), ('relationship', 'age'), ('relationship', 'income'), ('relationship', 'hours-per-week'), ('capital-gain', 'capital-loss'), ('native-country', 'race'), ('income', 'capital-loss'), ('income', 'capital-gain')]

Cantidad de relaciones:
26


La estructura obtenida automáticamente mediante Hill Climb Search contiene 30 relaciones entre las variables. A continuación, se estimarán los parámetros de esta segunda red utilizando el mismo conjunto de datos empleado para la red manual.

In [ ]:
from pgmpy.causal_discovery import PC

pc = PC(
    ci_test="chi_square",
    return_type="dag",
    show_progress=False
)

pc.fit(df)

estructura_auto = pc.causal_graph_

In [100]:

modelo_auto = DiscreteBayesianNetwork()

modelo_auto.add_nodes_from(df.columns)
modelo_auto.add_edges_from(estructura_auto.edges())

In [101]:
estimador_auto = MaximumLikelihoodEstimator(modelo_auto, df)

cpds_auto = estimador_auto.get_parameters()

modelo_auto.add_cpds(*cpds_auto)

/tmp/ipykernel_223051/2613678924.py:1: FutureWarning: `pgmpy.estimators.MaximumLikelihoodEstimator` is deprecated and will be removed in v1.3.0. Please use `pgmpy.parameter_estimator.DiscreteMLE` instead.
  estimador_auto = MaximumLikelihoodEstimator(modelo_auto, df)


In [102]:
print("¿Modelo automático válido?:", modelo_auto.check_model())
print("Cantidad de CPDs:", len(modelo_auto.get_cpds()))
print("Cantidad de relaciones:", len(modelo_auto.edges()))

¿Modelo automático válido?: True
Cantidad de CPDs: 13
Cantidad de relaciones: 41


## 1.5 Comparación entre la red manual y la red automática

La Red Bayesiana manual fue construida a partir de dependencias propuestas previamente, obteniendo una estructura de 13 nodos y 12 relaciones.

Por otra parte, la estructura aprendida automáticamente mediante el algoritmo PC contiene los mismos 13 nodos, pero presenta 41 relaciones.

Esto muestra que la red automática obtuvo una estructura considerablemente más densa que la red manual.

In [103]:
from pgmpy.inference import VariableElimination

inferencia_auto = VariableElimination(modelo_auto)

In [104]:
# Inferencia 1 - Red automática
resultado1_auto = inferencia_auto.query(
    variables=["income"],
    evidence={
        "education": "Bachelors",
        "hours-per-week": "Mas_40"
    }
)

print("INFERENCIA 1 - RED AUTOMÁTICA")
print(resultado1_auto)

INFERENCIA 1 - RED AUTOMÁTICA
+---------------+---------------+
| income        |   phi(income) |
+===============+===============+
| income(<=50K) |        0.4614 |
+---------------+---------------+
| income(>50K)  |        0.5386 |
+---------------+---------------+


In [105]:
# Inferencia 2 - Red automática
resultado2_auto = inferencia_auto.query(
    variables=["income"],
    evidence={
        "age": "Edad_1"
    }
)

print("INFERENCIA 2 - RED AUTOMÁTICA")
print(resultado2_auto)

INFERENCIA 2 - RED AUTOMÁTICA
+---------------+---------------+
| income        |   phi(income) |
+===============+===============+
| income(<=50K) |        0.9561 |
+---------------+---------------+
| income(>50K)  |        0.0439 |
+---------------+---------------+


In [106]:
# Inferencia 3 - Red automática
resultado3_auto = inferencia_auto.query(
    variables=["hours-per-week"],
    evidence={
        "age": "Edad_4"
    }
)

print("INFERENCIA 3 - RED AUTOMÁTICA")
print(resultado3_auto)

INFERENCIA 3 - RED AUTOMÁTICA
+--------------------------+-----------------------+
| hours-per-week           |   phi(hours-per-week) |
+==========================+=======================+
| hours-per-week(40_horas) |                0.4581 |
+--------------------------+-----------------------+
| hours-per-week(Mas_40)   |                0.3104 |
+--------------------------+-----------------------+
| hours-per-week(Menos_40) |                0.2314 |
+--------------------------+-----------------------+


### Comparación de las inferencias

Con el objetivo de comparar ambas redes, se realizaron las mismas tres consultas utilizando la Red Bayesiana manual y la Red Bayesiana cuya estructura fue aprendida automáticamente mediante PC.

| Inferencia | Evidencia | Red manual | Red automática |
|---|---|---:|---:|
| `P(income >50K)` | `Bachelors`, `Mas_40` | 42,69 % | 53,86 % |
| `P(income >50K)` | `Edad_1` | 21,28 % | 4,39 % |
| `P(hours-per-week = Mas_40)` | `Edad_4` | 30,16 % | 31,04 % |

### Comparación del ajuste mediante BIC

Para evaluar el ajuste de ambas estructuras sobre el mismo conjunto de datos se utilizó el criterio BIC (Bayesian Information Criterion).

Los resultados obtenidos fueron:

| Modelo | Relaciones | BIC |
|---|---:|---:|
| Red manual | 12 | -622.103,41 |
| Red automática (PC) | 41 | -169.170.082,51 |

En la puntuación BIC utilizada, un valor mayor indica una mejor relación entre el ajuste a los datos y la complejidad del modelo. La red manual obtuvo un BIC de **-622.103,41**, mientras que la red automática obtuvo **-169.170.082,51**.

In [107]:
from pgmpy.estimators import BIC

bic = BIC(df)

bic_manual = bic.score(modelo_manual)
bic_auto = bic.score(modelo_auto)

print("BIC red manual:", bic_manual)
print("BIC red automática:", bic_auto)

BIC red manual: -622103.4125560246
BIC red automática: -169170082.50777912


Según el criterio BIC calculado, la red manual presenta una puntuación superior a la red automática.

# Parte 2: Cadenas de Markov y Modelos Ocultos de Markov (HMM)

En esta parte se utilizará el dataset **Appliances Energy Prediction**, que contiene mediciones del consumo energético de una vivienda cada 10 minutos.

La variable `Appliances` se dividirá en seis niveles de consumo: muy bajo, bajo, medio-bajo, medio-alto, alto y muy alto.

Para las observaciones del HMM se utilizarán la variable `lights` y dos índices calculados a partir de las variables de temperatura y humedad:

$$
I_{\text{temperatura}} = \frac{T_1 + T_2 + T_3}{3}
$$

$$
I_{\text{humedad}} = \frac{RH_1 + RH_2 + RH_3}{3}
$$

La temperatura, la humedad y `lights` se clasificarán en tres niveles cada una, obteniendo un total de:

$$
3 \times 3 \times 3 = 27
$$

observaciones posibles.

**Uso de IA:** se utilizó una herramienta de inteligencia artificial como apoyo para consultar dudas sobre la implementación en Python y comprender algunos pasos del procedimiento. Las decisiones del desarrollo y la interpretación de los resultados fueron realizadas por el grupo.

## 2.a. Definición de estados, observaciones y separación de datos

In [108]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("energydata_complete.csv")

df.head()


,date,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,...,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2
0,2016-01-11 17:00:00,60,30,19.89,47.596667,19.2,44.790000,19.79,44.730000,19.000000,...,17.033333,45.53,6.600000,733.5,92.0,7.000000,63.000000,5.3,13.275433,13.275433
1,2016-01-11 17:10:00,60,30,19.89,46.693333,19.2,44.722500,19.79,44.790000,19.000000,...,17.066667,45.56,6.483333,733.6,92.0,6.666667,59.166667,5.2,18.606195,18.606195
2,2016-01-11 17:20:00,50,30,19.89,46.300000,19.2,44.626667,19.79,44.933333,18.926667,...,17.000000,45.50,6.366667,733.7,92.0,6.333333,55.333333,5.1,28.642668,28.642668
3,2016-01-11 17:30:00,50,40,19.89,46.066667,19.2,44.590000,19.79,45.000000,18.890000,...,17.000000,45.40,6.250000,733.8,92.0,6.000000,51.500000,5.0,45.410389,45.410389
4,2016-01-11 17:40:00,60,40,19.89,46.333333,19.2,44.530000,19.79,45.000000,18.890000,...,17.000000,45.40,6.133333,733.9,92.0,5.666667,47.666667,4.9,10.084097,10.084097


In [109]:
df.columns

Index(['date', 'Appliances', 'lights', 'T1', 'RH_1', 'T2', 'RH_2', 'T3',
       'RH_3', 'T4', 'RH_4', 'T5', 'RH_5', 'T6', 'RH_6', 'T7', 'RH_7', 'T8',
       'RH_8', 'T9', 'RH_9', 'T_out', 'Press_mm_hg', 'RH_out', 'Windspeed',
       'Visibility', 'Tdewpoint', 'rv1', 'rv2'],
      dtype='str')

In [110]:
df.shape

(19735, 29)

In [111]:
df["date"] = pd.to_datetime(df["date"])

df = df.sort_values("date").reset_index(drop=True)

df[["date", "Appliances", "lights"]].head()

,date,Appliances,lights
0,2016-01-11 17:00:00,60,30
1,2016-01-11 17:10:00,60,30
2,2016-01-11 17:20:00,50,30
3,2016-01-11 17:30:00,50,40
4,2016-01-11 17:40:00,60,40


Se calcularon dos índices usando los datos de temperatura y humedad.

Para la temperatura se hizo el promedio de `T1`, `T2` y `T3`:

$$
I_{\text{temperatura}} = \frac{T_1+T_2+T_3}{3}
$$

Para la humedad se hizo el promedio de `RH_1`, `RH_2` y `RH_3`:

$$
I_{\text{humedad}} = \frac{RH_1+RH_2+RH_3}{3}
$$

Estos valores, junto con `lights`, se usarán para crear las observaciones del HMM.

In [112]:
df["I_temperatura"] = (df["T1"] + df["T2"] + df["T3"]) / 3

df["I_humedad"] = ( df["RH_1"] + df["RH_2"] + df["RH_3"]) / 3

df[
    [
        "date",
        "Appliances",
        "lights",
        "I_temperatura",
        "I_humedad"
    ]
].head()

,date,Appliances,lights,I_temperatura,I_humedad
0,2016-01-11 17:00:00,60,30,19.626667,45.705556
1,2016-01-11 17:10:00,60,30,19.626667,45.401944
2,2016-01-11 17:20:00,50,30,19.626667,45.286667
3,2016-01-11 17:30:00,50,40,19.626667,45.218889
4,2016-01-11 17:40:00,60,40,19.626667,45.287778


In [113]:
df["Appliances"].describe()

count    19735.000000
mean        97.694958
std        102.524891
min         10.000000
25%         50.000000
50%         60.000000
75%        100.000000
max       1080.000000
Name: Appliances, dtype: float64

### Discretización de `Appliances`

La variable `Appliances` tiene muchos valores concentrados en consumos bajos y pocos valores muy altos.

Para dividirla en los seis estados de consumo se revisó su distribución y se utilizaron cortes basados en cuantiles, ajustándolos para evitar que un mismo valor quedara en categorías diferentes.

Los estados utilizados fueron:

- Muy bajo
- Bajo
- Medio-bajo
- Medio-alto
- Alto
- Muy alto

**Uso de IA:** se consultó una herramienta de inteligencia artificial para entender distintas formas de discretizar los datos y esta sugirió considerar cuantiles debido a la distribución desbalanceada de `Appliances`. A partir de esa orientación, el grupo revisó los datos y definió los límites utilizados.

In [114]:
bins_consumo = [
    -np.inf,
    40,
    50,
    60,
    80,
    120,
    np.inf
]

etiquetas_estados = [
    "Muy bajo",
    "Bajo",
    "Medio-bajo",
    "Medio-alto",
    "Alto",
    "Muy alto"
]

df["estado_consumo"] = pd.cut(
    df["Appliances"],
    bins=bins_consumo,
    labels=etiquetas_estados
)

df[["Appliances", "estado_consumo"]].head(20)

,Appliances,estado_consumo
0,60,Medio-bajo
1,60,Medio-bajo
2,50,Bajo
3,50,Bajo
4,60,Medio-bajo
5,50,Bajo
6,60,Medio-bajo
7,60,Medio-bajo
8,60,Medio-bajo
9,70,Medio-alto


In [115]:
df["estado_consumo"].value_counts().sort_index()

estado_consumo
Muy bajo      3094
Bajo          4368
Medio-bajo    3282
Medio-alto    2765
Alto          3231
Muy alto      2995
Name: count, dtype: int64

In [116]:
df.groupby(
    "estado_consumo",
    observed=True
)["Appliances"].agg(["min", "max", "count"])

,min,max,count
estado_consumo,,,
Muy bajo,10,40,3094
Bajo,50,50,4368
Medio-bajo,60,60,3282
Medio-alto,70,80,2765
Alto,90,120,3231
Muy alto,130,1080,2995


In [117]:
df["nivel_temperatura"] = pd.qcut(
    df["I_temperatura"],
    q=3,
    labels=["Bajo", "Medio", "Alto"]
)

df["nivel_humedad"] = pd.qcut(
    df["I_humedad"],
    q=3,
    labels=["Bajo", "Medio", "Alto"]
)

df[[
    "I_temperatura",
    "nivel_temperatura",
    "I_humedad",
    "nivel_humedad"
]].head(10)

,I_temperatura,nivel_temperatura,I_humedad,nivel_humedad
0,19.626667,Bajo,45.705556,Alto
1,19.626667,Bajo,45.401944,Alto
2,19.626667,Bajo,45.286667,Alto
3,19.626667,Bajo,45.218889,Alto
4,19.626667,Bajo,45.287778,Alto
5,19.626667,Bajo,45.153333,Alto
6,19.626667,Bajo,45.055556,Alto
7,19.595556,Bajo,44.986667,Alto
8,19.573333,Bajo,44.940278,Alto
9,19.625556,Bajo,45.117778,Alto


In [118]:
df["lights"].value_counts().sort_index()

lights
0     15252
10     2212
20     1624
30      559
40       77
50        9
60        1
70        1
Name: count, dtype: int64

In [119]:
def clasificar_lights(x):
    if x == 0:
        return "Apagada"
    elif x <= 20:
        return "Bajo"
    else:
        return "Alto"

df["nivel_lights"] = df["lights"].apply(clasificar_lights)

df[["lights", "nivel_lights"]].head(20)

,lights,nivel_lights
0,30,Alto
1,30,Alto
2,30,Alto
3,40,Alto
4,40,Alto
5,40,Alto
6,50,Alto
7,50,Alto
8,40,Alto
9,40,Alto


In [120]:
df["nivel_lights"].value_counts()

nivel_lights
Apagada    15252
Bajo        3836
Alto         647
Name: count, dtype: int64

In [121]:
df["observacion"] = (
    df["nivel_temperatura"].astype(str) + "_" +
    df["nivel_humedad"].astype(str) + "_" +
    df["nivel_lights"].astype(str)
)

df[[
    "nivel_temperatura",
    "nivel_humedad",
    "nivel_lights",
    "observacion"
]].head(15)

,nivel_temperatura,nivel_humedad,nivel_lights,observacion
0,Bajo,Alto,Alto,Bajo_Alto_Alto
1,Bajo,Alto,Alto,Bajo_Alto_Alto
2,Bajo,Alto,Alto,Bajo_Alto_Alto
3,Bajo,Alto,Alto,Bajo_Alto_Alto
4,Bajo,Alto,Alto,Bajo_Alto_Alto
5,Bajo,Alto,Alto,Bajo_Alto_Alto
6,Bajo,Alto,Alto,Bajo_Alto_Alto
7,Bajo,Alto,Alto,Bajo_Alto_Alto
8,Bajo,Alto,Alto,Bajo_Alto_Alto
9,Bajo,Alto,Alto,Bajo_Alto_Alto


In [122]:
df["observacion"].nunique()

27

In [123]:
df["observacion"].value_counts()

observacion
Bajo_Bajo_Apagada      1954
Medio_Medio_Apagada    1951
Alto_Bajo_Apagada      1939
Alto_Alto_Apagada      1924
Bajo_Medio_Apagada     1782
Alto_Medio_Apagada     1475
Medio_Bajo_Apagada     1434
Medio_Alto_Apagada     1422
Bajo_Alto_Apagada      1371
Medio_Alto_Bajo         653
Alto_Alto_Bajo          545
Bajo_Bajo_Bajo          484
Bajo_Medio_Bajo         445
Medio_Medio_Bajo        444
Medio_Bajo_Bajo         392
Bajo_Alto_Bajo          333
Alto_Medio_Bajo         287
Alto_Bajo_Bajo          253
Medio_Alto_Alto         141
Alto_Alto_Alto           96
Bajo_Alto_Alto           90
Medio_Medio_Alto         87
Bajo_Medio_Alto          72
Medio_Bajo_Alto          61
Bajo_Bajo_Alto           48
Alto_Medio_Alto          38
Alto_Bajo_Alto           14
Name: count, dtype: int64

In [124]:
indice_corte = int(len(df) * 0.8)

train = df.iloc[:indice_corte].copy()
test = df.iloc[indice_corte:].copy()

print("Entrenamiento:", train.shape)
print("Evaluación:", test.shape)

print("\nRango entrenamiento:")
print(train["date"].min(), "->", train["date"].max())

print("\nRango evaluación:")
print(test["date"].min(), "->", test["date"].max())

Entrenamiento: (15788, 36)
Evaluación: (3947, 36)

Rango entrenamiento:
2016-01-11 17:00:00 -> 2016-04-30 08:10:00

Rango evaluación:
2016-04-30 08:20:00 -> 2016-05-27 18:00:00


### Resultado de la preparación de datos

El conjunto de datos fue odernado cronológicamente, donde se construyeron los índices de temperatura y humedad a partir del promedio de cada una variables `T1`, `T2`, `T3` y `RH_1`, `RH_2`, `RH_3`. La variable `Appliances` fue discretizada en seis estados de consumo anteriormente mencionados.

Por otra parte, las variables utilizadas para construir las observaciones del HMM fueron discretizadas en tres niveles. A partir de la combinación de temperatura, humedad y `lights` se obtuvieron las 27 observaciones (que son el total de observaciones requeridas). 

Los registros fueron separados cronológicamente en un 80% para entrenamiento y un 20% para evaluación, comprendido por 15.788 registros, mientras que el conjunto de evaluación contiene 3.947 registros. El período de entrenamiento abarca desde el 11 de enero de 2016 hasta el 30 de abril de 2016, mientras que el conjunto de evaluación comprende desde el 30 de abril de 2016 hasta el 27 de mayo de 2016.

## 2.b. Construcción de las Markov chains

A partir de los estados de consumos obtenidos en el item anterior, se construirán dos Markov chains a partir del conjunto de entrenamiento, la primera cadena considerará todos los registros como una única secuencia continua, donde cada transición ocurre entre el último registro de un día y el primero al día siguiente, y la segunda cadena considerará cada día como una secuencia independiente, por lo que no se contabilizarán transiciones entre diferentes días.

### Construcción de la cadena continua

Lo que se quiere obtener es la matriz de transmisión, la cual funciona en determinar como de probable es cambiar de un estado, con la propiedad que la suma de cada fila tiene que ser 1. 

In [125]:
matriz_conteos_continua = pd.DataFrame(0, index=etiquetas_estados, columns=etiquetas_estados)

secuencia = train["estado_consumo"].astype(str).tolist()

for i in range(len(secuencia) - 1):
    estado_actual = secuencia[i]
    estado_siguiente = secuencia[i + 1]

    matriz_conteos_continua.loc[estado_actual, estado_siguiente] +=1

matriz_transicion_continua = (matriz_conteos_continua.div(matriz_conteos_continua.sum(axis=1), axis=0))

matriz_transicion_continua

,Muy bajo,Bajo,Medio-bajo,Medio-alto,Alto,Muy alto
Muy bajo,0.573753,0.302285,0.083795,0.026662,0.005540,0.007964
Bajo,0.259621,0.443711,0.225158,0.050546,0.010052,0.010913
Medio-bajo,0.104027,0.334312,0.345218,0.171560,0.018876,0.026007
Medio-alto,0.026962,0.099663,0.207511,0.431391,0.174771,0.059701
Alto,0.005596,0.013989,0.029177,0.161871,0.643485,0.145883
Muy alto,0.003667,0.010187,0.012225,0.046455,0.176447,0.751019


In [126]:
matriz_transicion_continua.sum(axis=1)

Muy bajo      1.0
Bajo          1.0
Medio-bajo    1.0
Medio-alto    1.0
Alto          1.0
Muy alto      1.0
dtype: float64

### Distribución estacionaria

In [127]:
P = matriz_transicion_continua.values

valores, vectores = np.linalg.eig(P.T)
indice = np.argmin(np.abs(valores - 1))
pi = np.real(vectores[:, indice])
pi = pi / pi.sum()

distribucion_estacionaria = pd.Series(
    pi,
    index=etiquetas_estados
)

distribucion_estacionaria

Muy bajo      0.182807
Bajo          0.220407
Medio-bajo    0.150855
Medio-alto    0.131561
Alto          0.158620
Muy alto      0.155750
dtype: float64

### Markov chain diaria

En este segundo modelo, cada día se considera como una secuencia independiente, por lo que no se contabilizan transiciones entre el último registro de un día y el primer registro del día siguiente. Esta distribución inicial se estimará utilizando el primer estado observado en cada jornada.

In [128]:
train["dia"] = train["date"].dt.date

matriz_conteos_diaria = pd.DataFrame(0, index=etiquetas_estados, columns=etiquetas_estados)

for _, grupo_dia in train.groupby("dia"):
    secuencia_dia = grupo_dia["estado_consumo"].astype(str).tolist()

    for i in range(len(secuencia_dia) - 1):
        estado_actual = secuencia_dia[i]
        estado_siguiente = secuencia_dia[i + 1]

        matriz_conteos_diaria.loc[estado_actual,estado_siguiente] += 1

matriz_conteos_diaria

matriz_transicion_diaria = (matriz_conteos_diaria.div(matriz_conteos_diaria.sum(axis=1), axis=0))

matriz_transicion_diaria

,Muy bajo,Bajo,Medio-bajo,Medio-alto,Alto,Muy alto
Muy bajo,0.574676,0.301154,0.084295,0.026233,0.005596,0.008045
Bajo,0.259560,0.443801,0.225087,0.050406,0.010139,0.011008
Medio-bajo,0.105151,0.333333,0.343125,0.172840,0.019157,0.026394
Medio-alto,0.027145,0.099370,0.205041,0.432380,0.175957,0.060107
Alto,0.005598,0.013994,0.029188,0.161535,0.643743,0.145942
Muy alto,0.003669,0.010192,0.012230,0.046474,0.176519,0.750917


In [129]:
matriz_transicion_diaria.sum(axis=1)

Muy bajo      1.0
Bajo          1.0
Medio-bajo    1.0
Medio-alto    1.0
Alto          1.0
Muy alto      1.0
dtype: float64

Ahora vemos como podemos obtener los estados bajo observaciones ocultas:

In [130]:
primeros_estados = (
    train
    .groupby("dia")
    .first()["estado_consumo"]
)

distribucion_inicial_diaria = (
    primeros_estados
    .value_counts(normalize=True)
    .reindex(etiquetas_estados, fill_value=0)
)

distribucion_inicial_diaria

estado_consumo
Muy bajo      0.207207
Bajo          0.369369
Medio-bajo    0.306306
Medio-alto    0.108108
Alto          0.000000
Muy alto      0.009009
Name: proportion, dtype: float64

In [131]:
print(distribucion_inicial_diaria.sum())

1.0


## 2.c, Análisis de las Markov chains


### Análisis del modelo continuo

In [132]:
(distribucion_estacionaria * 100).round(2)

Muy bajo      18.28
Bajo          22.04
Medio-bajo    15.09
Medio-alto    13.16
Alto          15.86
Muy alto      15.58
dtype: float64

Al calcular las probabilidades el consumo de energía se encuentra en el estado inicial es $q_0 = \text{Bajo}$. Por lo que se puede preguntar ¿Cual son las probabilidades dentro de una hora dado que el consumo inicial es bajo? Esto corresponde a calcular 6 transiciones dado este estado inicial, ya que las mediciones tienen un intervalo de 10 minutos.

In [133]:
P_1hora = np.linalg.matrix_power(matriz_transicion_continua.values, 6)

consulta_1hora = pd.Series(P_1hora[etiquetas_estados.index("Bajo")], index=etiquetas_estados)

consulta_1hora

Muy bajo      0.247558
Bajo          0.283424
Medio-bajo    0.178644
Medio-alto    0.119745
Alto          0.091290
Muy alto      0.079339
dtype: float64

Al cabo de una hora, el modelo muestra que el estado más probale sigue siendo $q_6 = \text{Bajo}$ con un $22.04\%$. Esto se puede interpretar que el estado inicial mantiene valores aproximados para el cálculo de los diferentes estados futuros. 

Para ver si el modelo sigue manteniendo los mismos resultados, podemos preguntar ¿Cual son las probabilidades dentro de tres horas dado que el consumo inicial es bajo?

In [134]:
P_3horas = np.linalg.matrix_power(matriz_transicion_continua.values, 18)

consulta_3horas = pd.Series(P_3horas[etiquetas_estados.index("Bajo")], index=etiquetas_estados)

consulta_3horas

Muy bajo      0.193006
Bajo          0.230367
Medio-bajo    0.155305
Medio-alto    0.129838
Alto          0.148134
Muy alto      0.143349
dtype: float64

Ahora al cabo de tres horas, las proabilidades se empienzan a aproximarse a la distribución estacionaria. Por ejemplo, la probabilidad de estar en estado `Bajo` disminuye desde $28.34\%$ a aproximadamente $23.04\%$, acercándose al valor estacionario de $22.04\%$. Esto muestra que, a medida que aumenta el horizonte temporal, la influencia del estado inicial disminuye y el sistema tiende hacia su comportamiento de largo plazo.

### Análisis del modelo diario

In [135]:
estados_iniciales_dia = (
    train
    .groupby("dia")
    .first()["estado_consumo"]
)

estados_finales_dia = (
    train
    .groupby("dia")
    .last()["estado_consumo"]
)

dist_inicial_dia = (
    estados_iniciales_dia
    .value_counts(normalize=True)
    .reindex(etiquetas_estados, fill_value=0)
)

dist_final_dia = (
    estados_finales_dia
    .value_counts(normalize=True)
    .reindex(etiquetas_estados, fill_value=0)
)

print("Distribución inicial:")
print((dist_inicial_dia * 100).round(2))

print("\nDistribución final:")
print((dist_final_dia * 100).round(2))

Distribución inicial:
estado_consumo
Muy bajo      20.72
Bajo          36.94
Medio-bajo    30.63
Medio-alto    10.81
Alto           0.00
Muy alto       0.90
Name: proportion, dtype: float64

Distribución final:
estado_consumo
Muy bajo      26.13
Bajo          27.03
Medio-bajo    31.53
Medio-alto    12.61
Alto           0.90
Muy alto       1.80
Name: proportion, dtype: float64


### Interpretación de los estados iniciales y finales

En la distribución inicial del modelo diario se observa que el estado predominante es `Bajo`, con un $36.94\%$, seguido de `Medio-bajo`, con un $30.63%$. Esto indica que la mayoría de las jornadas comienzan con niveles de consumo relativamente bajos.

En la distribución final, el estado más frecuente es `Medio-bajo`, con un $31.53\%$, seguido de `Bajo`, con un $27,03\%$, y `Muy bajo`, con un $26.13\%$.

En comparación con el inicio del día, al final de la jornada aumenta la proporción de estados `Muy bajo` y disminuye la del estado `Bajo`, además que los estados `Alto` y `Muy alto` son muy poco frecuentes al inicio como al final de cada día, lo que indica que los consumos elevados ocurren principalmente en los momentos intermedios de cada día.

Ahora si el cosnumo se encuentra bajo ¿Cómo este se distribuye después de dos horas?

In [136]:
P_diaria_2h = np.linalg.matrix_power(matriz_transicion_diaria.values, 12)

consulta_diaria_2h = pd.Series(P_diaria_2h[etiquetas_estados.index("Bajo")], index=etiquetas_estados)

consulta_diaria_2h

Muy bajo      0.208355
Bajo          0.244070
Medio-bajo    0.160849
Medio-alto    0.127450
Alto          0.133500
Muy alto      0.125775
dtype: float64

### Interpretación de las consultas del modelo diario

Partiendo desde el estado `Bajo`, después de dos horas la mayor probabilidad corresponde a permanecer o volver al estado `Bajo`, con aproximadamente un $24.41\%$, seguida por `Muy bajo`, con $20.84\%$.

Ahora otra consulta puede ser como ¿¿Cómo este se distribuye después de seis horas?

In [137]:
P_diaria_6h = np.linalg.matrix_power(matriz_transicion_diaria.values, 36)

consulta_diaria_6h = pd.Series(P_diaria_6h[etiquetas_estados.index("Bajo")],index=etiquetas_estados)

consulta_diaria_6h

Muy bajo      0.183134
Bajo          0.219575
Medio-bajo    0.149916
Medio-alto    0.131738
Alto          0.159399
Muy alto      0.156238
dtype: float64

Al extender el horizonte a seis horas, la distribución se vuelve más equilibrada entre los distintos estados. La probabilidad de encontrarse en un estdado `Bajo` es aproximadamente $21,96\%$, mientras que las probabilidades de estado de mayor consumo, como `Alto` y `Muy alto`, aumentan en valores. Esto muestra que, a medida que avanza el día, la influencia del estado inicial disminuye y el consumo actual puede evolucionar hacia distintos niveles con probabilidades similares.

## 2.d. Matriz de emisión y construcción de los modelos HMM

Ahora se construye la matriz de emisión, que representa la probabilidad de observar una determinada combinación de temperatura, humedad e iluminación, dado un estado de consumo, ya que esta matriz estima las probabilidades utilizando únicamente el conjunto de entrenamiento, mediante conteos y frecuencias relativas.

En priomer lugar, vamos a fijar el orden de las observaciones, ya que después cada columna de la matriz de emisión, esta corresponderá siempre a la misma observación.

In [138]:
observaciones = sorted(df["observacion"].unique())

print("Número de observaciones:", len(observaciones))

observaciones

Número de observaciones: 27


['Alto_Alto_Alto',
 'Alto_Alto_Apagada',
 'Alto_Alto_Bajo',
 'Alto_Bajo_Alto',
 'Alto_Bajo_Apagada',
 'Alto_Bajo_Bajo',
 'Alto_Medio_Alto',
 'Alto_Medio_Apagada',
 'Alto_Medio_Bajo',
 'Bajo_Alto_Alto',
 'Bajo_Alto_Apagada',
 'Bajo_Alto_Bajo',
 'Bajo_Bajo_Alto',
 'Bajo_Bajo_Apagada',
 'Bajo_Bajo_Bajo',
 'Bajo_Medio_Alto',
 'Bajo_Medio_Apagada',
 'Bajo_Medio_Bajo',
 'Medio_Alto_Alto',
 'Medio_Alto_Apagada',
 'Medio_Alto_Bajo',
 'Medio_Bajo_Alto',
 'Medio_Bajo_Apagada',
 'Medio_Bajo_Bajo',
 'Medio_Medio_Alto',
 'Medio_Medio_Apagada',
 'Medio_Medio_Bajo']

In [139]:
obs_a_numero = {
    obs: i
    for i, obs in enumerate(observaciones)
}

numero_a_obs = {
    i: obs
        for obs, i in obs_a_numero.items()
}

obs_a_numero

{'Alto_Alto_Alto': 0,
 'Alto_Alto_Apagada': 1,
 'Alto_Alto_Bajo': 2,
 'Alto_Bajo_Alto': 3,
 'Alto_Bajo_Apagada': 4,
 'Alto_Bajo_Bajo': 5,
 'Alto_Medio_Alto': 6,
 'Alto_Medio_Apagada': 7,
 'Alto_Medio_Bajo': 8,
 'Bajo_Alto_Alto': 9,
 'Bajo_Alto_Apagada': 10,
 'Bajo_Alto_Bajo': 11,
 'Bajo_Bajo_Alto': 12,
 'Bajo_Bajo_Apagada': 13,
 'Bajo_Bajo_Bajo': 14,
 'Bajo_Medio_Alto': 15,
 'Bajo_Medio_Apagada': 16,
 'Bajo_Medio_Bajo': 17,
 'Medio_Alto_Alto': 18,
 'Medio_Alto_Apagada': 19,
 'Medio_Alto_Bajo': 20,
 'Medio_Bajo_Alto': 21,
 'Medio_Bajo_Apagada': 22,
 'Medio_Bajo_Bajo': 23,
 'Medio_Medio_Alto': 24,
 'Medio_Medio_Apagada': 25,
 'Medio_Medio_Bajo': 26}

In [140]:
matriz_conteos_emision = pd.crosstab(
    train["estado_consumo"],
    train["observacion"]
)

matriz_conteos_emision = matriz_conteos_emision.reindex(
    index=etiquetas_estados,
    columns=observaciones,
    fill_value=0
)

matriz_conteos_emision

observacion,Alto_Alto_Alto,Alto_Alto_Apagada,Alto_Alto_Bajo,Alto_Bajo_Alto,Alto_Bajo_Apagada,Alto_Bajo_Bajo,Alto_Medio_Alto,Alto_Medio_Apagada,Alto_Medio_Bajo,Bajo_Alto_Alto,...,Bajo_Medio_Bajo,Medio_Alto_Alto,Medio_Alto_Apagada,Medio_Alto_Bajo,Medio_Bajo_Alto,Medio_Bajo_Apagada,Medio_Bajo_Bajo,Medio_Medio_Alto,Medio_Medio_Apagada,Medio_Medio_Bajo
estado_consumo,,,,,,,,,,,,,,,,,,,,,
Muy bajo,0,55,4,0,49,3,0,45,5,1,...,67,5,288,68,3,170,18,1,233,17
Bajo,5,151,15,0,124,3,0,125,5,3,...,75,6,427,82,0,286,27,7,553,40
Medio-bajo,7,91,38,0,123,9,0,116,12,17,...,61,28,242,97,11,238,59,10,413,61
Medio-alto,6,97,60,0,120,17,2,118,38,8,...,55,18,168,93,11,212,89,9,232,86
Alto,24,187,142,1,139,34,14,155,56,14,...,72,32,116,157,24,253,119,28,261,138
Muy alto,38,166,146,2,107,48,8,112,75,47,...,115,52,181,156,12,147,73,32,173,97


In [141]:
matriz_conteos_emision.shape

matriz_emision = (matriz_conteos_emision.div(matriz_conteos_emision.sum(axis=1), axis=0))

matriz_emision

observacion,Alto_Alto_Alto,Alto_Alto_Apagada,Alto_Alto_Bajo,Alto_Bajo_Alto,Alto_Bajo_Apagada,Alto_Bajo_Bajo,Alto_Medio_Alto,Alto_Medio_Apagada,Alto_Medio_Bajo,Bajo_Alto_Alto,...,Bajo_Medio_Bajo,Medio_Alto_Alto,Medio_Alto_Apagada,Medio_Alto_Bajo,Medio_Bajo_Alto,Medio_Bajo_Apagada,Medio_Bajo_Bajo,Medio_Medio_Alto,Medio_Medio_Apagada,Medio_Medio_Bajo
estado_consumo,,,,,,,,,,,,,,,,,,,,,
Muy bajo,0.000000,0.019044,0.001385,0.000000,0.016967,0.001039,0.000000,0.015582,0.001731,0.000346,...,0.023199,0.001731,0.099723,0.023546,0.001039,0.058864,0.006233,0.000346,0.080679,0.005886
Bajo,0.001436,0.043366,0.004308,0.000000,0.035612,0.000862,0.000000,0.035899,0.001436,0.000862,...,0.021539,0.001723,0.122631,0.023550,0.000000,0.082137,0.007754,0.002010,0.158817,0.011488
Medio-bajo,0.002936,0.038171,0.015940,0.000000,0.051594,0.003775,0.000000,0.048658,0.005034,0.007131,...,0.025587,0.011745,0.101510,0.040688,0.004614,0.099832,0.024748,0.004195,0.173238,0.025587
Medio-alto,0.002889,0.046702,0.028888,0.000000,0.057776,0.008185,0.000963,0.056813,0.018296,0.003852,...,0.026481,0.008666,0.080886,0.044776,0.005296,0.102070,0.042850,0.004333,0.111700,0.041406
Alto,0.009592,0.074740,0.056755,0.000400,0.055556,0.013589,0.005596,0.061950,0.022382,0.005596,...,0.028777,0.012790,0.046363,0.062750,0.009592,0.101119,0.047562,0.011191,0.104317,0.055156
Muy alto,0.015479,0.067617,0.059470,0.000815,0.043585,0.019552,0.003259,0.045621,0.030550,0.019145,...,0.046843,0.021181,0.073727,0.063544,0.004888,0.059878,0.029735,0.013035,0.070468,0.039511


Ahora cada entrada de la matriz representa la probabilidad de una observación dado un estado ($P(O_j| S_i)$), lo que también representa una observación aparece más según el estado de consumo correspondiente. 

In [142]:
matriz_emision.sum(axis=1)

estado_consumo
Muy bajo      1.0
Bajo          1.0
Medio-bajo    1.0
Medio-alto    1.0
Alto          1.0
Muy alto      1.0
dtype: float64

Ahora generamos el modelo HMM, con lo realizado en el apartado 2.b

In [143]:
from hmmlearn.hmm import CategoricalHMM

hmm_continuo = CategoricalHMM(n_components=6, init_params="", params="")

hmm_continuo.startprob_ = distribucion_estacionaria.values
hmm_continuo.transmat_ = matriz_transicion_continua.values
hmm_continuo.emissionprob_ = matriz_emision.values

hmm_continuo.n_features = len(observaciones)

print("Distribución inicial:", hmm_continuo.startprob_.shape)
print("Matriz transición:", hmm_continuo.transmat_.shape)
print("Matriz emisión:", hmm_continuo.emissionprob_.shape)

Distribución inicial: (6,)
Matriz transición: (6, 6)
Matriz emisión: (6, 27)


In [144]:
hmm_diario = CategoricalHMM(n_components=6, init_params="", params="")

hmm_diario.startprob_ = distribucion_inicial_diaria.values
hmm_diario.transmat_ = matriz_transicion_diaria.values
hmm_diario.emissionprob_ = matriz_emision.values
hmm_diario.n_features = len(observaciones)

print("Distribución inicial:", hmm_diario.startprob_.shape)
print("Matriz transición:", hmm_diario.transmat_.shape)
print("Matriz emisión:", hmm_diario.emissionprob_.shape)

Distribución inicial: (6,)
Matriz transición: (6, 6)
Matriz emisión: (6, 27)


### Diferencias entre los dos HMM:

La principal diferencia entre ambos modelos HMM se encuentra en la forma en que se representan el inicio de la secuencia y las transiciones entre estados. En el HMM continuo, la distribución inicial corresponde a la distribución estacionaria de la Markov chain, donde las transiciones se estiman considerando todos los registros como una única secuencia, junto a las transiciones entre el último registro de un día y el primero del día siguiente. En cambio, en el HMM diario, la distribución inicial se construye con el primer estado observado de cada jornada, y las transiciones se calculan únicamente dentro de cada día, dejando afuera así las transiciones entre días diferentes.

Ambos modelos utilizan la misma matriz de emisión, ya que esta representa la probabilidad de observar una determinada combinación de temperatura, humedad e iluminación dado un estado de consumo, relación que se estima a partir del mismo conjunto de entrenamiento.

## 2.e Inferencia sobre un día del conjunto de evaluación

En esta sección analizaremos que ocurrirá durante un día dado los modelos descritos anteriormente.

In [145]:
test["dia"] = test["date"].dt.date

test.groupby("dia").size()

dia
2016-04-30     94
2016-05-01    144
2016-05-02    144
2016-05-03    144
2016-05-04    144
2016-05-05    144
2016-05-06    144
2016-05-07    144
2016-05-08    144
2016-05-09    144
2016-05-10    144
2016-05-11    144
2016-05-12    144
2016-05-13    144
2016-05-14    144
2016-05-15    144
2016-05-16    144
2016-05-17    144
2016-05-18    144
2016-05-19    144
2016-05-20    144
2016-05-21    144
2016-05-22    144
2016-05-23    144
2016-05-24    144
2016-05-25    144
2016-05-26    144
2016-05-27    109
dtype: int64

Dado que las mediciones se registran cada 10 minutos, un día completo contiene 144 registros, por este motivo es que se escogerá una fecha con 144 observaciones, evitando los días incompletos del inicio y final del conjunto de evaluación. Elegimos Para este análisis se utilizará el $2016-05-01$.

In [146]:
dia_elegido = pd.to_datetime("2016-05-01").date()

datos_dia = test[test["dia"] == dia_elegido].copy()

datos_dia.shape

(144, 37)

In [147]:
datos_dia[
    [
        "date",
        "Appliances",
        "estado_consumo",
        "observacion"
    ]
].head()

,date,Appliances,estado_consumo,observacion
15882,2016-05-01 00:00:00,50,Bajo,Medio_Medio_Apagada
15883,2016-05-01 00:10:00,60,Medio-bajo,Medio_Medio_Apagada
15884,2016-05-01 00:20:00,50,Bajo,Medio_Medio_Apagada
15885,2016-05-01 00:30:00,50,Bajo,Medio_Medio_Apagada
15886,2016-05-01 00:40:00,60,Medio-bajo,Medio_Medio_Apagada


In [148]:
datos_dia["obs_num"] = datos_dia["observacion"].map(obs_a_numero)

X_dia = datos_dia["obs_num"].values.reshape(-1, 1)

X_dia.shape

(144, 1)

### Forward:

A partir del algóritmo Fordward se pueden calcular los estados siguentes dada una observación inicial, donde al tener 144 observaciones, cada probabilidad se vuelva muy pequeña, y gracias a la librería `hmmlearn`, esta entrega la log-probabilidad de la secuencia, y así calcular las probabilidades correctamente.


In [149]:
log_prob_continuo = hmm_continuo.score(X_dia)
log_prob_diario = hmm_diario.score(X_dia)

print("Log-probabilidad modelo continuo:", round(log_prob_continuo,2))
print("Log-probabilidad modelo diario:", round(log_prob_diario, 2))

Log-probabilidad modelo continuo: -373.22
Log-probabilidad modelo diario: -372.8


### Interpretación de Forward

El modelo continuo obtuvo una log-probabilidad de aproximadamente $-373.22$, mientras que el modelo diario obtuvo una log-probabilidad de aproximadamente $-372.8$. 

Una log-probabilidad mayor corresponde a una secuencia más probable bajo el modelo, el HMM diario representa ligeramente mejor las observaciones del día seleccionado. La diferencia entre ambos valores es pequeña, sin embargo (en este día particular), cada jornada es interpretada como una secuencia independiente de acuerdo a un ajuste levemente superior.

### Fordward-Backward:

Ahora con el algóritmo Fordward-Backward se puede obtener la probabilidad de cada estado en cualquier lugar de la secuencia.

In [150]:
logprob_cont, post_cont = hmm_continuo.score_samples(X_dia)
logprob_dia, post_dia = hmm_diario.score_samples(X_dia)

print(post_cont.shape)
print(post_dia.shape)

(144, 6)
(144, 6)


Cada fila corresponde a uno de los 144 instantes del día y cada columna a uno de los 6 estados. Luego podemos responder consultas en momentos concretos, por ejemplo, a las 08:00, 12:00 y 18:00. Como el día comienza a las 00:00:

In [151]:
indices_consulta = {
    "08:00": 48,
    "12:00": 72,
    "18:00": 108
}

for hora, idx in indices_consulta.items():
    print(f"\nHora {hora}.")

    print("\nModelo continuo:")
    print(pd.Series(post_cont[idx], index=etiquetas_estados))

    print("\nModelo diario:")
    print(pd.Series(post_dia[idx], index=etiquetas_estados))


Hora 08:00.

Modelo continuo:
Muy bajo      0.133437
Bajo          0.355582
Medio-bajo    0.302291
Medio-alto    0.138461
Alto          0.048707
Muy alto      0.021522
dtype: float64

Modelo diario:
Muy bajo      0.133521
Bajo          0.354763
Medio-bajo    0.302873
Medio-alto    0.138136
Alto          0.049020
Muy alto      0.021688
dtype: float64

Hora 12:00.

Modelo continuo:
Muy bajo      0.004545
Bajo          0.032102
Medio-bajo    0.077370
Medio-alto    0.226946
Alto          0.459499
Muy alto      0.199538
dtype: float64

Modelo diario:
Muy bajo      0.004514
Bajo          0.031767
Medio-bajo    0.076516
Medio-alto    0.227411
Alto          0.460338
Muy alto      0.199455
dtype: float64

Hora 18:00.

Modelo continuo:
Muy bajo      0.004084
Bajo          0.029117
Medio-bajo    0.071865
Medio-alto    0.220143
Alto          0.464743
Muy alto      0.210048
dtype: float64

Modelo diario:
Muy bajo      0.004061
Bajo          0.028850
Medio-bajo    0.071164
Medio-alto    0.220704
Al

Por ejemplo, a las 08:00, el modelo continuo asigna la mayor probabilidad al estado `Bajo`, con aproximadamente $35.56%$, seguido de `Medio-bajo`, con $30.23\%$. Los estados `Alto` y `Muy alto` presentan probabilidades considerablemente menores, esto se debe a que disponer de todos los datos del día, el modelo considera más probable que el consumo a esa hora se encuentre en niveles bajos o medio-bajos.

Ahora aplicamos Fordward-Backward para determinar los estados a las 08:00, a las 12:00 y a las 18:00:

In [152]:
for hora, idx in indices_consulta.items():
    print(f"\nHora {hora}.")

    cont = pd.Series(post_cont[idx], index=etiquetas_estados)
    dia = pd.Series(post_dia[idx], index=etiquetas_estados)

    print("\nModelo Continuo:", cont.idxmax(),
          f"({cont.max()*100:.2f}%)")

    print("Modelo Diario:", dia.idxmax(),
          f"({dia.max()*100:.2f}%)")

    print("Estado real:",
          datos_dia.iloc[idx]["estado_consumo"])


Hora 08:00.

Modelo Continuo: Bajo (35.56%)
Modelo Diario: Bajo (35.48%)
Estado real: Medio-bajo

Hora 12:00.

Modelo Continuo: Alto (45.95%)
Modelo Diario: Alto (46.03%)
Estado real: Alto

Hora 18:00.

Modelo Continuo: Alto (46.47%)
Modelo Diario: Alto (46.54%)
Estado real: Muy alto


### Interpretación de Forward-Backward

Se puede ver que a las 08:00, tanto el modelo continuo como el modelo diario obtienen el estado `Bajo`, con aproximadamente $35.5\%$. Sin embargo, el estado real corresponde a `Medio-bajo`. A las 12:00, ambos modelos identifican correctamente el estado `Alto`, con probabilidades cercanas al $46.0\%$, coincidiendo con el estado real observado. A las 18:00, ambos modelos asignan la mayor probabilidad al estado `Alto`, mientras que el estado real corresponde a `Muy alto`. 

Los modelos difieren con respecto a los estado reales, esto se debe a que el analizar una secuencia continua de estados por medio de un HMM puede no llevar a los mismos estados reales. En general, los resultados de ambos HMM son muy similares en los instantes analizados, lo que indica que la diferencia entre considerar una secuencia continua o jornadas independientes no produce cambios importantes en estas consultas particulares.

### Viterbi

Utilizando el algóritmo de Viterbi junto a la log-probailidad para determinar la secuencia más probable

In [153]:
log_viterbi_cont, estados_viterbi_cont = hmm_continuo.decode(X_dia, algorithm="viterbi")

log_viterbi_dia, estados_viterbi_dia = hmm_diario.decode(X_dia, algorithm="viterbi")

print("Log-probabilidad Viterbi continuo:", log_viterbi_cont)
print("Log-probabilidad Viterbi diario:", log_viterbi_dia)

Log-probabilidad Viterbi continuo: -425.5243281574236
Log-probabilidad Viterbi diario: -424.9124745414572


In [154]:
viterbi_cont_nombres = [etiquetas_estados[i] for i in estados_viterbi_cont]
viterbi_dia_nombres = [etiquetas_estados[i] for i in estados_viterbi_dia]

comparacion_viterbi = pd.DataFrame({
    "date": datos_dia["date"].values,
    "real": datos_dia["estado_consumo"].astype(str).values,
    "viterbi_continuo": viterbi_cont_nombres,
    "viterbi_diario": viterbi_dia_nombres
})

comparacion_viterbi.head(20)

,date,real,viterbi_continuo,viterbi_diario
0,2016-05-01 00:00:00,Bajo,Bajo,Bajo
1,2016-05-01 00:10:00,Medio-bajo,Bajo,Bajo
2,2016-05-01 00:20:00,Bajo,Bajo,Bajo
3,2016-05-01 00:30:00,Bajo,Bajo,Bajo
4,2016-05-01 00:40:00,Medio-bajo,Bajo,Bajo
5,2016-05-01 00:50:00,Muy bajo,Bajo,Bajo
6,2016-05-01 01:00:00,Medio-bajo,Bajo,Bajo
7,2016-05-01 01:10:00,Bajo,Bajo,Bajo
8,2016-05-01 01:20:00,Medio-bajo,Bajo,Bajo
9,2016-05-01 01:30:00,Bajo,Bajo,Bajo


In [155]:
accuracy_cont = (comparacion_viterbi["real"] == comparacion_viterbi["viterbi_continuo"]).mean()

accuracy_dia = (comparacion_viterbi["real"] == comparacion_viterbi["viterbi_diario"]).mean()

print(f"Exactitud Viterbi continuo: {accuracy_cont*100:.2f}%")
print(f"Exactitud Viterbi diario: {accuracy_dia*100:.2f}%")

Exactitud Viterbi continuo: 38.89%
Exactitud Viterbi diario: 38.89%


### Interpretación de Viterbi

El modelo continuo obtuvo una log-probabilidad de aproximadamente $-425.52$, mientras que el modelo diario obtuvo aproximadamente $-424.91$. Como un valor mayor representa una secuencia más probable, el modelo diario presenta una leve ventaja en este criterio.

Al comparar las secuencias inferidas con los estados reales del consumo, ambos modelos alcanzaron una probabilidad de $38.89\%$. Esto indica que para el día analizado ambos HMM producen una secuencia de estados similares, presentan la misma capacidad de acierto con respecto los estados reales, pero con la diferencia de considerar una secuencia continua o separar las jornadas no genera una mejora importante en la exactitud obtenida mediante el algórtimo.

## 2.f. Comparación e interpretación de los modelos

### Comparación entre el modelo continuo y el modelo diario

El modelo continuo toma todos los registros como una sola secuencia, por lo que también considera el cambio entre el último registro de un día y el primero del día siguiente.

En cambio, el modelo diario separa cada día y solo considera las transiciones que ocurren dentro de una misma jornada.

Por esta razón, las matrices de transición de ambos modelos son distintas. El modelo continuo incluye cambios entre días, mientras que el modelo diario se enfoca solamente en cómo cambia el consumo dentro de cada jornada.

### Significado de las distribuciones iniciales

Los modelos también usan distribuciones iniciales diferentes.

En el modelo continuo se utiliza la distribución estacionaria de la cadena de Markov, que representa el comportamiento esperado del sistema a largo plazo.

En el modelo diario, la distribución inicial se calculó usando el primer estado de cada día. Esto permite ver con qué nivel de consumo suele comenzar una jornada.

## Conclusión

En general, ambos modelos HMM presentan comportamientos similares sobre el día evaluado. El modelo diario obtiene log-probabilidades ligeramente superiores tanto en Forward como en Viterbi, pero la diferencia respecto del modelo continuo es significativamente pequeña, y ambos alcanzan la misma exactitud.

El modelo continuo incorpora información proveniente de las transiciones entre días y utiliza una distribución inicial de largo plazo, mientras que el modelo diario representa de manera más específica el comienzo y la evolución de cada jornada.

La elección entre ambos modelos depende, por lo tanto, de la interpretación temporal que se desea realizar, si el objetivo es analizar una serie de eventos continuos, el modelo continuo resulta ser el más apropiado, pero si se desea estudiar el comportamiento del consumo dentro dentro de un día en particular, el modelo diario puede adaptarse más para conseguir resultados correctos.